In [0]:

df_gold = spark.read.table("Staging_Table")

display(df_gold)


In [0]:
df_condition = df_gold.groupBy("Medical_Condition").count()

display(df_condition)

print(f"Total row: {df_condition.count()}")


Databricks visualization. Run in Databricks to view.

In [0]:
import matplotlib.pyplot as plt

df_condition_pd = df_condition.toPandas()

bar_colors = ['tab:red', 'tab:green', 'tab:blue', 'tab:orange', 'tab:purple', 'tab:brown']
ax = df_condition_pd.plot.line(x='Medical_Condition', y='count', rot=0 ,color=bar_colors, figsize=(10, 6))

ax.set_ylabel('Total Patient')
ax.set_xlabel('Medical Condition')
ax.set_title('Distribution of Medical Condition')
ax.legend (title ='count')
plt.show()

In [0]:
from pyspark.sql.functions import col, count, avg, round
df_gold2 = df_gold.groupby('Medical_Condition').agg(count(col('Patient_ID')).alias("Total_Patient"), round(avg('Age'), 1).alias('Average_Age')).show()

In [0]:
from pyspark.sql.functions import col, count, avg
df_insurance = df_gold.groupBy('Insurance_Provider').agg(count(col('Patient_ID')).alias("Total_Patient"), avg('Billing_Amount').alias('Average_Billing')).orderBy(col('Total_Patient').desc()).show()

display(df_insurance)


In [0]:
df_disease = df_gold.groupBy('Medical_Condition','Blood_Type').agg(count(col('Patient_ID')).alias("Total_Patient")).orderBy(col('Total_Patient').desc()).show()

display(df_disease)



In [0]:

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df_corr = df_gold.toPandas()

df_corr['Billing_Amount'] = df_corr['Billing_Amount'].astype(float)

correlation = df_corr.corr(method='pearson',numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(
    correlation, 
    annot=True,          
    cmap="coolwarm",   
    vmin=-1, vmax=1,   
    linewidths=0.5
)

plt.title("Correlation Heatmap")
plt.show()

In [0]:
df_gold.write.format("delta").mode("overwrite").saveAsTable("healthcare.hospital_ml_data")

## Data Warehouse (STAR SCHEMA)
- Dimension Table 
 1. Dim_patient  : Patient_ID(Primary Key), Age, Gender, Blood Type
 2. Dim_Hospital : Hospital_ID(Primary Key), Hospital, Room Number
 3. Dim_Doctor : Doctor_ID, Doctor
 4. Dim Insurance: Insurance_ID (Primary Key), Insurance_Provider
 5. Dim_Clinical : Clinical_ID(Primary Key), Medical Condition, Admission Type, Medication, Test Result

- Fact Table
 1. Patient_ID (Foreign Key)
 2. Hospital_ID (Foreign Key)
 3. Insurance_ID (Foreign Key)
 4. Clinical_ID (Foreign Key)
 5. Doctor_ID
 6. Billing Amount
 7. Date of Admission
 8. Discharge Date
 9. Estimated Length of Stay
 10. File name (Metadata for audit)
 11. Ingest time (Metadata for audit)


In [0]:
from pyspark.sql.functions import monotonically_increasing_id, col

dim_patient = df_gold.select("Patient_ID","Age","Gender", "Blood_Type").distinct()

dim_patient.write.mode("overwrite").format("delta").saveAsTable("Dim_Patient")

display(dim_patient)

In [0]:
dim_hospital = df_gold.select("Hospital", "Room_Number").distinct() \
.withColumn("Hospital_id", monotonically_increasing_id())

dim_hospital.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("Dim_Hospital")

display(dim_hospital)

In [0]:
dim_doctor = df_gold.select("Doctor").distinct() \
.withColumn("Doctor_id", monotonically_increasing_id())

dim_doctor.write.mode("overwrite").format("delta").saveAsTable("Dim_Doctor")

display(dim_doctor)

In [0]:
dim_insurance = df_gold.select("Insurance_Provider").distinct() \
.withColumn("Insurance_id", monotonically_increasing_id())

dim_insurance.write.mode("overwrite").format("delta").saveAsTable("Dim_Insurance")

display(dim_insurance)

In [0]:
dim_clinical = df_gold.select("Medical_Condition", "Medication", "Test_Results","Admission_Type").distinct() \
.withColumn("Clinical_id", monotonically_increasing_id())

dim_clinical.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("Dim_Clinical")

display(dim_clinical)

In [0]:
fact_table = df_gold \
    .join(dim_patient, on="Patient_ID", how="left") \
    .join(dim_hospital, on=["Hospital", "Room_Number"], how="left") \
    .join(dim_doctor, on="Doctor", how="left") \
    .join(dim_insurance, on="Insurance_Provider", how="left") \
    .join(dim_clinical, on=["Medical_Condition", "Medication", "Test_Results", "Admission_Type"], how="left") \
    .select(col("Patient_ID"),
        col("Hospital_Id"),
        col("Doctor_Id"),
        col("Insurance_Id"),
        col("Clinical_Id"),
        col("Date_of_Admission"),
        col("Discharge_Date"),
        col("Billing_Amount"),
        col("Estimated_Length_of_Stay")
    )

fact_table.write.mode("overwrite").format("delta").option("overwriteschema", "true").saveAsTable("Fact_Medical_Billing")

display(fact_table)

In [0]:
fact_table.count()
print(f"Total row of records in Fact Table: {fact_table.count()}")